In [ ]:
#Część 1

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

In [ ]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

In [ ]:
df.show(10, truncate=False)

In [ ]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

In [ ]:
#Część 2

#2.1

from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

In [ ]:
#2.2

from pyspark.sql.functions import min as _min, max as _max

category_stats = (
    df.groupBy("category")
    .agg(
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(_min("amount"), 2).alias("min_PLN"),
        _round(_max("amount"), 2).alias("max_PLN"),
    )
    .orderBy("category")
)

category_stats.show(truncate=False)

In [ ]:
#Część 3

#3.1

from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

In [ ]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

In [ ]:
#3.2

window_30_store = (
    df.groupBy(
        window("timestamp", "30 minutes"),
        "store"
    )
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        col("store"),
        col("liczba_tx"),
        col("suma_PLN"),
    )
    .orderBy("od", "store")
)

window_30_store.show(truncate=False)

In [ ]:
#3.3

from pyspark.sql.functions import desc

krakow_top_hour = (
    df.filter(col("store") == "Kraków")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        col("suma_PLN"),
    )
    .orderBy(desc("suma_PLN"))
)

krakow_top_hour.show(1, truncate=False)

In [ ]:
#Część 4

#4.1

sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

In [ ]:
#4.2

tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

#Sliding ma więcej wierszy, bo okna się nakładają i są tworzone częściej. Jedna transakcja może trafić do więcej niż jednego okna.

In [ ]:
#Część 5

#1. 4661
#2. groupBy("store") = wynik łączny per sklep; groupBy(window(...), "store") = wynik per sklep w każdym oknie czasowym.
#3. 2 okna 

In [ ]:
#Praca domowa

from pyspark.sql.functions import avg

gdansk_lowest_avg = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(avg("amount"), 2).alias("srednia_PLN")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "srednia_PLN",
    )
    .orderBy("srednia_PLN")
)

gdansk_lowest_avg.show(1, truncate=False)

In [ ]:
from pyspark.sql.functions import hour, minute

window_0900_0930 = df.filter(
    (hour(col("timestamp")) == 9) &
    (minute(col("timestamp")) < 30)
)

window_0900_0930.show(5, truncate=False)

In [ ]:
category_0900_0930 = (
    window_0900_0930
    .groupBy("category")
    .agg(
        count("tx_id").alias("liczba_tx")
    )
    .orderBy("category")
)

category_0900_0930.show(truncate=False)

In [ ]:
peak_15min = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
    )
    .orderBy(desc("liczba_tx"))
)

peak_15min.show(1, truncate=False)